# Making word embeddings with BPEmb

## Load Dataset

In [20]:
from collections import Counter
from pathlib import Path
from scipy.sparse import csr_matrix
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import random
from sympy import true

import part1 # to use code from tutorial
import torch

def set_seed(seed):
    """Seeds Python, NumPy (also used by sklearn's saga solver) and PyTorch (weight init, shuffling, dropout)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

In [21]:
# splitting data into train and test sets and folds for cross-validation
LABEL_COL = "Party_orientation"
SAMPLE_FRAC = 1.0  # fraction of the data to use, 1.0 for all of it

# get the dataframe: one dataframe with tokens, lemmas, sentences and embeddings,
# so all three representations are trained and evaluated on the same rows
df = pd.read_pickle("./merged_embeddings_sampled.pkl")
df = df.sample(frac=SAMPLE_FRAC, random_state=42).reset_index(drop=True)

# train test split
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df[LABEL_COL])
train_df = train_df.reset_index(drop=True)

# labels and settings
label_names = sorted(df[LABEL_COL].unique())
outcome_space = part1.clslib.NamedClasses(label_names)
# repeated cross-validation: each seed gives its own fold split and model initialisation
# (fixed in advance; more runs give the Bayesian correlated t-test more paired differences to work with)
my_seeds = [0, 1, 2]
num_folds = 5

## Loading Embedding Models

## BPEmb

In [22]:
from bpemb import BPEmb

bpemb_hu = BPEmb(lang="hu", vs=100000)

## fastText

Download fastText vectors

In [40]:
!curl -L https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.hu.300.vec.gz -o cc.hu.300.vec.gz

bind: Key with name 'btab' does not have any mapping
bind: Key with name 'btab' does not have any mapping
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1199M  100 1199M    0     0  4851k      0  0:04:13  0:04:13 --:--:-- 7834k 0:01:11 11.3M3.2M      0  0:01:30  0:00:38  0:00:52 9915k 0:01:02 1177k0  0:02:38  0:01:29  0:01:09 1398k53  0:01:20 1018k  0:01:57  0:01:22  994k01:19 5084k2  0:02:15  0:01:17 4910k01:14 1578k  0  5194k      0  0:03:56  0:02:42  0:01:14 1865k1:13 2654k 0  5434k      0  0:03:45  0:02:54  0:00:51 13.9M03:38  0:02:57  0:00:41 16.7M35  0:03:09  0:00:26 5473k  0:00:15 1581k 0  0:03:54  0:03:40  0:00:14 1881k4 1444k


In [45]:
from gensim.models import KeyedVectors

fasttext_hu = KeyedVectors.load_word2vec_format("cc.hu.300.vec.gz", limit=500_000)

In [31]:
from collections import Counter

pair_counts, token_counts, lemma_counts = Counter(), Counter(), Counter()

TOKEN_COL, LEMMA_COL = "tokens", "lemmas"
token_rows = train_df[TOKEN_COL]
lemma_rows = train_df[LEMMA_COL]

for tokens, lemmas in zip(token_rows, lemma_rows):
    token_counts.update(tokens)
    lemma_counts.update(lemmas)
    pair_counts.update(zip(tokens, lemmas))

print(f"Unique Tokens: {len(token_counts)}")
print(f"Unique Lemmas: {len(lemma_counts)}")
print(f"Unique Pairs: {len(pair_counts)}")

Unique Tokens: 355300
Unique Lemmas: 150674
Unique Pairs: 370954


# Defining Helper Functions

In [85]:
def bpemb_valid_ids(word, model):
    if not word:
        return None

    ids = model.encode_ids(word)

    if not ids or model.spm.unk_id() in ids:
        return None

    return ids

def bpemb_word_vector(word, model):
    # model is BPEmb object
    ids = bpemb_valid_ids(word, model)

    if ids is None:
        return None

    return model.emb[ids].mean(axis=0)

def fasttext_word_vector(word, model):
    # model is FastText KeyedVectors object
    if not word or word not in model.key_to_index:
        return None

    return model[word]

In [86]:
sample_word = train_df['tokens'][0][0]

print(f"BPEmb Word Vector Shape: {bpemb_word_vector(sample_word, bpemb_hu).shape}")
print(f"FastText Word Vector Shape: {fasttext_word_vector(sample_word, fasttext_hu).shape}")

BPEmb Word Vector Shape: (100,)
FastText Word Vector Shape: (300,)


# Checking dataset cases

In [87]:
for col in [TOKEN_COL, LEMMA_COL]:
    non_lowercase = [word for row in train_df[col] for word in row if word != word.lower()]
    print(f"{col}: all lowercase = {not non_lowercase}")

tokens: all lowercase = True
lemmas: all lowercase = True


# Checking fastText coverage

In [88]:
from collections import Counter

for col in [TOKEN_COL, LEMMA_COL]:
    word_counts = Counter(word for row in train_df[col] for word in row)

    covered = sum(
        count for word, count in word_counts.items()
        if fasttext_word_vector(word, fasttext_hu) is not None
    )
    print(f"FastText coverage for {col}: (as-is): {covered / sum(word_counts.values()):.3f}")

FastText coverage for tokens: (as-is): 0.961
FastText coverage for lemmas: (as-is): 0.970


# Checking Coverage

In [89]:
print(f"BPEmb <UNK> ID: {bpemb_hu.spm.unk_id()}")

BPEmb <UNK> ID: 0


In [97]:
from functools import partial
from collections import Counter

TOP_MISSING = 30

fasttext_check = partial(fasttext_word_vector, model=fasttext_hu)
bpemb_check = partial(bpemb_valid_ids, model=bpemb_hu)

coverage_checks = [("fastText", fasttext_check), ("BPEmb", bpemb_check)]

for name, check in coverage_checks:
    for col in [TOKEN_COL, LEMMA_COL]:
        word_counts = Counter(word for row in train_df[col] for word in row)
        total = sum(word_counts.values())

        is_covered = {word: check(word) is not None for word in word_counts}
        covered = sum(count for word, count in word_counts.items() if is_covered[word])
        missing = [
            (word, count)
            for word, count in word_counts.most_common()
            if not is_covered[word]
        ][:TOP_MISSING]

        print(f"{name} coverage for {col}: {covered / total}")
        print(f"{name} top {TOP_MISSING} missing for {col}: {missing}")
    print()

fastText coverage for tokens: 0.9614940336428206
fastText top 30 missing for tokens: [('magyarországot', 1449), ('magyarországnak', 1265), ('orbán-kormány', 1170), ('fidesz-kdnp', 877), ('xxi.', 838), ('képviselőtársaimat', 825), ('lőrinc', 641), ('képviselőtársunk', 587), ('brüsszel', 584), ('képviselőtársamnak', 571), ('§-a', 570), ('brüsszelben', 537), ('koronavírus-járvány', 534), ('kárpát', 462), ('magyarországért', 453), ('magyarországról', 441), ('fidesz-kormány', 431), ('rogán', 413), ('vezérszónoki', 406), ('ensz', 373), ('képviselőtársammal', 346), ('európát', 344), ('koronavírus', 338), ('képviselőtársak', 329), ('klik', 326), ('képviselőtársaimnak', 317), ('fidesznek', 316), ('fidesz-frakció', 300), ('mszp-s', 299), ('szijjártó', 298)]
fastText coverage for lemmas: 0.9702310315669149
fastText top 30 missing for lemmas: [('képviselőtárs', 12299), ('mindannyi', 1837), ('brüsszel', 1624), ('orbán-kormány', 1378), ('fidesz-kdnp', 924), ('igyekezik', 876), ('megteremtés', 875), 

## Getting suffixes and counts

In [ ]:
from collections import defaultdict

# ház -> házban works, we get -ban, but for example kéz -> kezet does not
def get_suffix(token, lemma):
    if token != lemma and token.startswith(lemma):
        return token[len(lemma):]

    return None

def collect_suffix_info(pair_counts, word_vector):
    info = defaultdict(list)
    vectors = {}

    def get_vector(word):
        if word not in vectors:
            vectors[word] = word_vector(word)

        return vectors[word]

    for (token, lemma), frequency in pair_counts.items():
        suffix = get_suffix(token, lemma)
        if suffix is None:
            continue

        if get_vector(token) is None or get_vector(lemma) is None:
            continue

        info[suffix].append((token, lemma, frequency))

    return dict(info), vectors

def offsets(pairs, vectors):
    token_vectors = np.stack([vectors[token] for token, _ in pairs])
    lemma_vectors = np.stack([vectors[lemma] for _, lemma in pairs])

    return token_vectors, lemma_vectors, token_vectors - lemma_vectors

embedding_checks = [
    ("fastText", partial(fasttext_word_vector, model=fasttext_hu)),
    ("BPEmb", partial(bpemb_word_vector, model=bpemb_hu)),
]

for name, word_vector in embedding_checks:
    suffix_info, vectors = collect_suffix_info(pair_counts, word_vector)
    print(f"{name} suffixes:", len(suffix_info))

fastText suffixes: 851
BPEmb suffixes: 1059


## Cos similarity and offset consistency

In [100]:
def unit(matrix):
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return np.divide(matrix, norms, out=np.zeros_like(matrix, dtype=float), where=norms != 0)


def token_lemma_cos(token_vectors, lemma_vectors):
    return np.mean(np.sum(unit(token_vectors) * unit(lemma_vectors), axis=1))


def mean_pairwise_cos(offset_vectors):
    if len(offset_vectors) < 2:
        return np.nan

    normalized = unit(offset_vectors)
    similarities = normalized @ normalized.T
    off_diagonal = similarities[~np.eye(len(normalized), dtype=bool)]
    return off_diagonal.mean()


def offsets(pairs, vectors):
    token_vectors = np.stack([vectors[token] for token, _ in pairs])
    lemma_vectors = np.stack([vectors[lemma] for _, lemma in pairs])
    return token_vectors, lemma_vectors, token_vectors - lemma_vectors


MIN_PAIRS = 50
MAX_SUFFIXES = 30
rng = np.random.default_rng(0)

rows = []

for name, word_vector in embedding_checks:
    suffix_info, vectors = collect_suffix_info(pair_counts, word_vector)

    suffixes = [
        suffix
        for suffix, suffix_pairs in suffix_info.items()
        if len(suffix_pairs) >= MIN_PAIRS
    ]
    suffixes = sorted(
        suffixes,
        key=lambda suffix: len(suffix_info[suffix]),
        reverse=True,
    )[:MAX_SUFFIXES]

    all_pairs = [
        (token, lemma)
        for suffix_pairs in suffix_info.values()
        for token, lemma, _ in suffix_pairs
    ]

    for suffix in suffixes:
        pairs = [
            (token, lemma)
            for token, lemma, _ in suffix_info[suffix]
        ]

        token_vectors, lemma_vectors, offset_vectors = offsets(pairs, vectors)

        random_indices = rng.choice(len(all_pairs), size=len(pairs), replace=False)
        random_pairs = [all_pairs[i] for i in random_indices]
        _, _, random_offsets = offsets(random_pairs, vectors)

        rows.append({
            "embedding": name,
            "suffix": suffix,
            "n": len(pairs),
            "cos(token, lemma)": token_lemma_cos(token_vectors, lemma_vectors),
            "offset consistency": mean_pairwise_cos(offset_vectors),
            "random baseline": mean_pairwise_cos(random_offsets),
        })

results = pd.DataFrame(rows).round(3)
print(results.to_string(index=False))

embedding suffix     n  cos(token, lemma)  offset consistency  random baseline
 fastText      t  4243              0.674               0.201            0.079
 fastText     ek  2156              0.687               0.213            0.079
 fastText     ok  1979              0.697               0.211            0.080
 fastText    nak  1939              0.653               0.168            0.081
 fastText      a  1859              0.692               0.225            0.079
 fastText    nek  1743              0.648               0.164            0.076
 fastText      e  1719              0.695               0.219            0.080
 fastText     ra  1405              0.611               0.258            0.080
 fastText     ni  1318              0.579               0.299            0.079
 fastText    ban  1283              0.645               0.187            0.081
 fastText     re  1245              0.606               0.251            0.078
 fastText     át  1211              0.565           